# An Open-Source Digital Digital Delay Locked Loop for **Educational** and **Architectural** Exploration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SiliconJackets/Cac_Spring26/blob/main/DelayLockedLoop.ipynb)

```
Copyright 2026 SiliconJackets @ Georgia Institute of Technology
SPDX-License-Identifier: Apache-2.0
```

|Name|Affiliation| Email |IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|:----------:|
|Ethan Huang|Georgia Institute of Technology|ethanhuang@gatech.edu|No|No|
|Zheng-Yin Lee|Georgia Institute of Technology|zlee63@gatech.edu|No|No|
|Alfi Misha Antony Selvin Raj|Georgia Institute of Technology|alfiselvin@gatech.edu|No|No|
|Oliver S. Lee|Georgia Institute of Technology|xli3086@gatech.edu|Yes|No|
|Shreyas Angadi|Georgia Institute of Technology|sangadi6@gatech.edu|No|No|
|Mythri Muralikannan|Georgia Institute of Technology|mmuralikannan3@gatech.edu|Yes|Yes|


<br>


---
## **Overview**
---
<br>

This notebook serves as an interactive, supplementary resource for students learning about Delay-Locked Loops (DLLs). It explores several design variations of the key parts (submodules) of a DLL, specifically the Phase Detector, Controller, and Digitally Controlled Delay Line (DCDL). For each component, qualitative insights into behavior and design tradeoffs are paired with quantitative results received from SPICE simulations.

<br>

This exploration culminates in an interactive implementation of a practical DLL as a Zero-Delay Buffer (ZDB), presented through two complementary approaches:
- A high-level Python model of each submodule derived from its SPICE-characterized behavior. This enables users to quickly swap between different design variants and observer their impact on overall system performance.
- A parameterizable System Verilog (RTL) implementation in which users can tune key parameters within a selected submodule and modify it's RTL-level behavior.

<br>

Simulating these changes allows users to view how design parameters influence the locking capabilities of the DLL. A Python interface additionally provides insights and feedback on the design choices made.

<br>

While Python-based simulations provide a fast and accessible way to explore the architectural trade-offs, RTL level behavioral and SPICE simulations remains essential for identifying implementation specific issues (glitches and race conditions) as well as parameter sensitivity (convergence behavior).

<br>

This design targets the [open source sky130 PDK](https://github.com/google/skywater-pdk/), a 130 nm CMOS process developed by Google and SkyWater Technology Foundry, supporting a fully open-source design flow from RTL through layout and fabrication.


<br>


---
## **Setup**
---

<br>

WARNING: This step may take several minutes to complte. Please run it once before proceeding with the rest of the notebook.

<br>

In [ ]:
# @title Download Dependencies {display-mode: "form"}
# @markdown
# @markdown Click the ▷ button to download all neccesary dependencies (librelane + nix-os)
# @markdown as well as all of our RTL files + scripts for this interactive notebook
# @markdown
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

# NgSpice Installation
!sudo apt-get update
!sudo apt-get install -y ngspice

# Nix Installation
os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"


# Librelane Installation
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

#Install our scripts
%cd /content/
!rm -rf CAC_2026
!git clone https://github.com/SiliconJackets/CaC_Spring26 CAC_2026

#Install iverilog
!sudo apt-get install -y iverilog

#Install VCD reader
!pip install vcdvcd -q

#Install Streamlit
!pip install streamlit


In [ ]:
# @title Import Our Repository and Plotting Framework {display-mode: "form"}
# @markdown
# @markdown Click the ▷ button to download all of our RTL files + scripts for this interactive notebook
# @markdown
# Setup all neccesary graphic design parts
import sys
import os
from pathlib import Path

# Define paths and resolve '~' to an absolute path
PROJECT_ROOT = Path("/content/CAC_2026")
SPICE_DIR = PROJECT_ROOT / "spice"
scripts_path = str(PROJECT_ROOT / "scripts")

# Add the scripts folder to the system path
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

from analysis_util import find_switching_time, get_sample, apply_time_shift, decode_ctrl, decode_q
from plot_framework import iplot, ioverlay, isweep, istack, isweep_overlay
from ngspice_loader import load_wrdata, load_sweep
from vcdvcd import VCDVCD


from bokeh.io import output_notebook
output_notebook()


<br>


---
## **Section 0: Fundamentals of Delay-Locked-Loops (DLLs)**
---

<br>

### **What is a Delay-Locked-Loop (DLL)?**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dll.gif?raw=true">
</p>

<br>

A Delay-Locked Loop (DLL) is a closed-loop feedback system used to precisely align the timing (phase) of one clock signal (`clk_out`) with another reference clock (`clk_in`). Rather than generating a new clock frequency, a DLL adjusts the timing of an existing clock by introducing a controllable delay.

<br>

At a high level, the goal of a DLL is simple: make the output clock edge occur at the exact same time as the reference clock edge. Importantly, the DLL does not track which specific clock cycle is being aligned; instead, it ensures that the edges are aligned in time, even if this corresponds to a delay of one or more full clock periods.

<br>


---

<br>

### **What Problem Does a DLL Solve?**

<br>

In real digital systems, clock signals do not arrive everywhere at the same time. As a clock propagates through buffers, wires, and logic, it accumulates delay due to:
- Wire resistance and capacitance
- Gate delays
- Process, voltage, and temperature (PVT) variations

This leads to clock skew, where different parts of a system see the clock at slightly different times.

<br>

DLLs are widely used in real systems, including:
- Clock distribution networks
- High-speed I/O interfaces (e.g., DDR, PCIe)
- Deskewing clock trees
- Zero-Delay Buffers (ZDBs)

<br>

In a Zero-Delay Buffer, the DLL ensures that the clock seen by downstream logic is perfectly aligned with the original reference clock.

<br>

A DLL operates as a feedback control system:
1. Compare the reference clock and output clock
2. Determine which one arrives first
3. Adjust the delay accordingly
4. Repeat until alignment is achieved

<br>

This continuous correction allows the DLL to track changes in the environment, such as:
- Temperature shifts
- Supply voltage fluctuations
- Manufacturing variations

This feedback allows the DLLs to maintain accurate timing even in non-ideal conditions.

<br>

---

<br>

### **The Three Core Components of a DLL**

<br>

A digital DLL is typically composed of three key building blocks:

#### 1. Phase Detector (PD)
The Phase Detector answers a simple but critical question:

> “Which clock edge came first?”

- If the **output clock is late**: increase delay  
- If the **output clock is early**: decrease delay  

Most digital DLLs use a **bang-bang phase detector** (also used here), which outputs a binary decision (early/late) rather than a continuous value.

This makes the system simple, robust, and well-suited for digital implementation  

<br>

#### 2. Controller
The Controller processes the output of the Phase Detector and decides how to update the delay.

Typical implementations include:
- Up/Down counters  
- Finite State Machines (FSMs)  

Conceptually, it is a simple counter using the logic: $\text{delay} = \text{delay} + (\text{up}-\text{down})$.

It acts as the **“brain” of the DLL**, determining how aggressively or smoothly the system converges to lock.

<br>

### 3. Digitally Controlled Delay Line (DCDL)
The DCDL is the component that actually **shifts the clock in time**. It introduces a programmable delay: $\text{Total Delay} = D_{min} + N\times \Delta d$.

Where:
- $D_{min}$ = minimum delay  
- $\Delta d$ = delay resolution (step size)  
- $N$ = digital control value  

Key design trade-offs:
- **Resolution ($\Delta d$):** smaller steps $\implies$ higher accuracy  
- **Range:** must cover at least one clock period  
- **Linearity:** delay should increase predictably  
- **Noise sensitivity:** affects jitter and stability  

<br>

---

<br>

### What Does “Lock” Mean?

<br>

A DLL is considered **locked** when:
- The rising edge of the output clock aligns with the rising edge of the reference clock  
- The delay line has converged to a stable value  

At lock:
- The system stops making large adjustments  
- Only small corrections occur to track noise and variation  

In practice, perfect alignment is not required; the DLL is considered locked when the phase error is within a small tolerance determined by the delay resolution and phase detector sensitivity.

In many designs, the total delay equals **one full clock period**, meaning: The delayed clock is effectively a phase-shifted version of the input.

<br>

---

<br>

### Delay Locked Loop vs Phase Locked Loop

<br>


| Feature          | DLL                   | PLL                        |
| ---------------- | --------------------- | -------------------------- |
| Output frequency | Same as input         | Can change frequency       |
| Control variable | Delay                 | Phase + frequency          |
| Complexity       | Lower                 | Higher                     |
| Stability        | Generally more stable | Can be harder to stabilize |

A DLL **does not generate a new clock frequency**. It only adjusts timing.

<br>

---

<br>

### What You’ll Explore in This Notebook

<br>

This notebook explores the design of a digital DLL through its key submodules, combining SPICE-based analysis with high-level modeling. You will examine how different design choices impact system behavior, ultimately building toward a complete DLL-based **Zero-Delay Buffer (ZDB)**.




<br>


---
## **Section 1: Exploring Architectural Trade-offs in DLL Components**
---

### **Section 1a: Phase Detector**

<br>

The **phase detector (PD)** is responsible for comparing the timing of the reference clock (`clk_in`) and the feedback clock (`clk_out`). Specifically, it observes the **rising edges** of both signals and determines which one occurs first.

<br>

Based on this comparison, the phase detector generates two single-bit outputs:
- **up** is asserted when the reference clock leads (increase delay).
- **down** is asserted when the feedback clock leads (decrease delay).  

<br>

Importantly, the phase detector does **not measure the magnitude** of the phase difference, only its **direction** (early or late). This makes it a coarse but fast decision-making block within the loop.

<br>
Most digital DLLs use a **bang-bang phase detector**, which outputs a binary decision rather than a continuous error signal. While this approach is simple and efficient, it introduces an important limitation: The loop can never settle perfectly at a fixed point.

<br>

Near lock, even very small timing differences cause the detector to alternate between `up` and `down`. This results in continuous small corrections, producing **limit-cycle behavior** and observable **jitter** in the output clock.

<br>

---

<br>

#### Architectural Trade-offs

<br>

The choice of phase detector architecture has a significant impact on overall DLL performance. Key considerations include:

- **Accuracy**: How precisely the detector can determine phase alignment  
- **Stability**: How smoothly the loop converges to lock  
- **Jitter**: How much oscillation occurs near lock  
- **Metastability robustness**: Reliability when edges arrive very close together  
- **Complexity**: Area, power, and implementation cost  

<br>

Simpler detectors (e.g., single flip-flop designs) offer low area and power with fast operation but suffer from bias and steady-state error and higher jitter near lock.  

<br>


More advanced detectors (e.g., Phase-Frequency Detectors) can improve robustness but introduce greater complexity and design constraints.

<br>

In the following sections, we explore several phase detector architectures, comparing their behavior, trade-offs, and performance using SPICE simulation results.

---



<br>


#### **Implementation 1: Single Flip-Flop Detector**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/pd.png?raw=true" width="500">
</p>

<br>

The **single flip-flop phase detector** is the simplest possible implementation. A D-type flip-flop samples the feedback clock (`clk_out`) on the rising edge of the reference clock (`clk_in`).

- If `clk_out` is **low** at the sampling instant → the feedback clock is **late** → `up` is asserted  
- If `clk_out` is **high** at the sampling instant → the feedback clock is **early** → `down` is asserted  

This detector always produces a decision; there is no explicit “equal” or neutral state.


##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for a Single Flip-Flop Detector (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_ff1

In [ ]:
# @markdown Change these parameters to run your own post-synthesis simulations for this phase detector (NOTE: Each spice simulation will take some time to run)

clk_in_delay = 20  # @param {key:"CLK_IN Delay From Start", type:"integer"}
clk_out_delay = 40  # @param {key:"CLK_OUT Delay From Start", type:"integer"}

!python CAC_2026/scripts/flow_cli.py phase_detector_syn_ff1 --process spice --clk-in-delay {clk_in_delay}n --clk-out-delay {clk_out_delay}n

file_name = f"phase_detector_syn_ff1_clkin{clk_in_delay}n_clkout{clk_out_delay}n.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title=f"FF1 phase detector — {clk_in_delay}ns delay for clk_in and {clk_out_delay}ns delay for clk_out",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 23), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 23)},
        {"RST": get_sample(traces["RST"], 19.5, 23)},
        {"UP": get_sample(traces["UP"], 19.5, 23), "DOWN": get_sample(traces["DOWN"], 19.5, 23)},
    ],
    title="FF1 phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)


**Graph Analysis**

In the SPICE waveform above, this behavior can be seen clearly: when `clk_out` rises before `clk_in`, the detector samples `clk_out = 1` at the rising edge of `clk_in`, resulting in `down = 1`. Because the detector only updates on `clk_in` edges and does not generate pulses, the output remains latched until the next sampling event.

**Advantages**
- **Extremely simple**: requires only a single flip-flop and minimal logic.
- **Low area and power**: smallest footprint of all implementations.  
- **High-speed operation**: very short critical path enables high frequency.  

**Limitations**
- **Inherent bias**: even at perfect alignment, the detector favors one direction, leading to a small steady-state phase offset.
- **No frequency detection**: cannot distinguish between phase error and frequency mismatch.
- **High jitter near lock**: continuous bang-bang toggling limits steady-state accuracy.

This detector is ideal for **minimal-area, high-speed designs**, but sacrifices accuracy and introduces systematic error in the locked state.

In [ ]:
# @markdown An example graph of clk_in leading.

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_ff1_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 23), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 23)},
        {"RST": get_sample(traces["RST"], 19.5, 23)},
        {"UP": get_sample(traces["UP"], 19.5, 23), "DOWN": get_sample(traces["DOWN"], 19.5, 23)},
    ],
    title="FF1 phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)


<br>


#### **Implementation 2: Edge-Order Detector**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/edgeorderpd.png?raw=true" width="500">
</p>

<br>

The **edge-order detector** improves upon the single flip-flop approach by explicitly determining which clock edge arrives first.

Each clock edge attempts to assert its corresponding output:
- `clk_in` rising edge asserts `up`  
- `clk_out` rising edge asserts `down`  

However, only the **first arriving edge wins**. The second edge clears the output, completing the comparison for that cycle.


##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for a Edge-Order Detector (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_edge

In [ ]:
# @markdown Change these parameters to run your own post-synthesis simulations for this phase detector (NOTE: Each spice simulation will take some time to run)

clk_in_delay = 20  # @param {key:"CLK_IN Delay From Start", type:"integer"}
clk_out_delay = 25  # @param {key:"CLK_OUT Delay From Start", type:"integer"}

!python CAC_2026/scripts/flow_cli.py phase_detector_syn_edge --process spice --clk-in-delay {clk_in_delay}n --clk-out-delay {clk_out_delay}n

file_name = f"phase_detector_syn_edge_clkin{clk_in_delay}n_clkout{clk_out_delay}n.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title=f"Edge phase detector — {clk_in_delay}ns delay for clk_in and {clk_out_delay}ns delay for clk_out",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkin_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 23), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 23)},
        {"RST": get_sample(traces["RST"], 19.5, 23)},
        {"UP": get_sample(traces["UP"], 19.5, 23), "DOWN": get_sample(traces["DOWN"], 19.5, 23)},
    ],
    title="Edge phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

**Graph Analysis**

<br>

In the waveforms above, this behavior is observed as a **latched decision that persists until the opposite edge arrives**. For example, when `clk_in` leads, `up` is asserted and remains high until the subsequent `clk_out` edge clears it. Similarly, when `clk_out` leads, `down` is asserted and held until `clk_in` arrives. This creates a pulse-like behavior that encodes the ordering of edges rather than simply sampling at a single instant.

**Advantages**
- **Low complexity**: only slightly more logic than the single flip-flop design.
- **More symmetric behavior**: reduces bias compared to simple sampling.  
- **No explicit reset loop required**: edges naturally clear each other.  

**Limitations**
- **Race conditions**: when edges arrive very close together, both `up` and `down` may briefly assert.  
- **Metastability sensitivity**: near-aligned edges can lead to unpredictable behavior.  
- **Incorrect decisions possible**: edge ordering may not always be captured reliably.  
- **No frequency detection capability**

This design improves conceptual correctness over the single flip-flop detector by tracking **edge ordering over time**, but remains limited by **timing ambiguity and metastability near lock**.



In [ ]:
# @markdown Example graph of clk_out leading
traces = load_wrdata(
    SPICE_DIR / "results" / "pd_syn_edge_clkout_lead_results.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 23), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 23)},
        {"RST": get_sample(traces["RST"], 19.5, 23)},
        {"UP": get_sample(traces["UP"], 19.5, 23), "DOWN": get_sample(traces["DOWN"], 19.5, 23)},
    ],
    title="Edge phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)


<br>


#### **Implementation 3: Phase-Frequency Detector (PFD)**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/pd/pfdpd.png?raw=true" width="500">
</p>

<br>

The **Phase-Frequency Detector (PFD)** is the most widely used phase detector architecture in practical designs.

It consists of two flip-flops initialized with `D = 1`:
- The rising edge of `clk_in` sets the first flip-flop and asserts `up`  
- The rising edge of `clk_out` sets the second flip-flop and asserts `down`  

When both outputs are high, an internal reset signal clears both flip-flops, completing the comparison cycle.

Unlike simpler detectors, the PFD **tracks both clock edges over time rather than sampling at a single instant**. The first arriving edge asserts its output, and that output remains active until the second edge arrives and resets the system.

##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for a Edge-Order Detector (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py phase_detector_syn_pfd

In [ ]:
# @markdown Change these parameters to run your own post-synthesis simulations for this phase detector (NOTE: Each spice simulation will take some time to run)

clk_in_delay = 20  # @param {key:"CLK_IN Delay From Start", type:"integer"}
clk_out_delay = 25  # @param {key:"CLK_OUT Delay From Start", type:"integer"}

!python CAC_2026/scripts/flow_cli.py phase_detector_syn_pfd --process spice --clk-in-delay {clk_in_delay}n --clk-out-delay {clk_out_delay}n

file_name = f"phase_detector_syn_pfd_clkin{clk_in_delay}n_clkout{clk_out_delay}n.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": traces["CLK_IN"], "CLK_OUT": traces["CLK_OUT"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
    ],
    title=f"PFD phase detector — {clk_in_delay}ns delay for clk_in and {clk_out_delay}ns delay for clk_out",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:
traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkout_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 40), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 40)},
        {"RST": get_sample(traces["RST"], 19.5, 40)},
        {"UP": get_sample(traces["UP"], 19.5, 40), "DOWN": get_sample(traces["DOWN"], 19.5, 40)},
    ],
    title="PFD phase detector — clk_out leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

**Graph Analysis**

<br>
In the waveform above, this behavior is visible as follows: when `clk_out` leads, `down` is asserted first. When `clk_in` eventually arrives, both signals briefly become high, triggering the reset and clearing both outputs. This creates a well-defined correction event for each cycle.

**Advantages**
- **Robust edge tracking**: correctly identifies lead/lag even when edges are close.
- **Symmetric behavior**: avoids the steady-state bias seen in simpler detectors.
- **Frequency detection capability**: sustained assertions occur if one clock consistently leads.
- **Widely used and well understood**: standard architecture in PLL/DLL systems.

**Limitations**
- **Higher complexity**: requires additional logic (two flip-flops + reset path).
- **Dead zone (practical)**: very small timing differences may produce very short events that are difficult to resolve.
- **Reset path constraints**: limits maximum operating frequency.

The PFD provides the most **robust and reliable phase detection** by tracking both clock edges and resolving their ordering over time. While it is more complex than simpler detectors, it avoids bias and behaves predictably even near lock, making it the **standard choice for high-performance designs**.

In [ ]:
# @markdown Example graph of clk_in leading

traces = load_wrdata(
    SPICE_DIR / "results" / "phase_detector_syn_pfd_clkin_leads.csv",
    ["CLK_IN", "CLK_OUT", "RST", "UP", "DOWN"],
)

istack(
    layers=[
        {"CLK_IN": get_sample(traces["CLK_IN"], 19.5, 40), "CLK_OUT": get_sample(traces["CLK_OUT"], 19.5, 40)},
        {"RST": get_sample(traces["RST"], 19.5, 40)},
        {"UP": get_sample(traces["UP"], 19.5, 40), "DOWN": get_sample(traces["DOWN"], 19.5, 40)},
    ],
    title="PFD phase detector — clk_in leads",
    xlabel="Time (ns)",
    ylabels=["Clocks (V)", "Reset (V)", "Outputs (V)"],
)

#### **Phase Detector Comparison**
| Detector | Accuracy | Jitter | Robustness | Complexity | Key Limitation |
|----------|----------|--------|------------|------------|----------------|
| Single FF | Low | High | Low | Very Low | Bias / no neutral state |
| Edge-Order | Medium | Medium | Medium | Low | Race conditions |
| PFD | High | Low | High | Medium | Reset path complexity |


<br>


### **Section 1b: Controller**

<br>

The **controller** receives the `up` and `down` signals from the phase detector and updates an N-bit control word (`ctrl`) that determines the delay of the DCDL.

<br>

At a high level, the controller translates the **directional decisions** of the phase detector into **discrete adjustments of delay**:
- `up = 1`: increase delay  
- `down = 1`: decrease delay  
- otherwise: hold value  

<br>

Unlike the phase detector, which determines *which direction to move*, the controller determines *how the system moves toward lock*. As a result, it plays a critical role in shaping the overall behavior of the loop.

<br>

Different controller architectures directly impact:
- how quickly the DLL reaches lock  
- how much it oscillates near lock  
- how robust it is to noise and small timing variations  

<br>

All controller designs balance a fundamental trade-off between **convergence speed** and **steady-state stability**, which is explored through the implementations below.



<br>


#### **Implementation 1: Saturating Controller**

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/saturating.png?raw=true" width="500">
</p>

<br>

The **saturating controller** is the baseline implementation.

On each clock cycle:
- `up = 1` $\implies$ `ctrl = ctrl + 1`  
- `down = 1` $\implies$ `ctrl = ctrl - 1`  

The control word is clamped to: $0\le \text{ctrl}\le 2^{N-1}$.

##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the Saturating Controller (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py controller_saturate

In [ ]:
# @markdown Click the ▷ button to run your own post-synthesis simulations for this saturating controller (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py controller_saturate --process spice

file_name = f"controller_saturate.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / "controller_saturate.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_saturate.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Saturating Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

**Graph Analysis**
| Phase                     | Time Range (ns) | Cycles | Inputs (UP / DOWN)               | ctrl Behavior              | What We Are Testing                                                                                                                    |
| ------------------------- | --------------- | ------ | -------------------------------- | -------------------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| **0: Reset**              | 0 – 4           | 2      | rst = 1                          | `ctrl → 32`                | **Initialization**: starts at the correct value after reset                                                                            |
| **1: Sustained UP**       | 4 – 44          | 20     | UP=1, DOWN=0                     | `32 → 52` (+20)            | **Increment behavior**: +1 step per cycle, no skips                                                                                    |
| **2: Idle**               | 44 – 64         | 10     | UP=0, DOWN=0                     | Hold at 52                 | **Hold behavior (idle)**: no change when inputs are 0                                                                                  |
| **3: Sustained DOWN**     | 64 – 104        | 20     | UP=0, DOWN=1                     | `52 → 32` (−20)            | **Decrement behavior**: −1 step per cycle, matches the UP behavior                                                                     |
| **4: Idle**               | 104 – 114       | 5      | UP=0, DOWN=0                     | Hold at 32                 | **Stability**: no drift or unintended transitions during idle                                                                          |
| **5: Alternating Inputs** | 114 – 134       | 10     | UP/DOWN alternate every 3 cycles | Ideally hold near constant | **Conflict/priority behavior**: tests condition `up = down` → HOLD. Checks if design incorrectly prioritizes one signal over the other |
| **6: Sustained UP**       | 134 – 174       | 20     | UP=1, DOWN=0                     | `~32 → ~52` (+20)          | **Recovery**: verifies controller resumes correct counting from held value after disturbances                                          |
| **7: Final Idle**         | 174 – 180       | 3      | UP=0, DOWN=0                     | Hold value                 | **Final stability check**: confirms steady-state holding at end of simulation                                                          |

**Advantages**
- Simple and predictable behavior  
- Minimal area and power  
- Easy to verify and reason about  

**Limitations**
- Slow convergence when far from lock (fixed step size)  
- Persistent limit-cycle jitter near lock  
- No adaptability to operating conditions  

A simple and robust baseline, but fundamentally limited by its fixed step size and unavoidable steady-state oscillation.



<br>


#### **Implementation 2: Filtered Controller**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/filter.png?raw=true" width="500">
</p>


<br>

The **filtered controller** reduces sensitivity to short-term fluctuations by requiring multiple consistent decisions before updating.

- Consecutive `up` or `down` requests are counted  
- Only after reaching a threshold (`FILTER_LEN`) is `ctrl` updated  
- Any direction change resets the counters  


##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the Filtered Controller (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py controller_filtered

In [ ]:
# @markdown Click the ▷ button to run your own post-synthesis simulations for this filtered controller (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py controller_filtered --process spice

file_name = f"controller_filtered.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Filtered Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_filtered.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Filtered Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

**Graph Analysis**
| Phase | Time Range (ns) | Cycles | Inputs (UP / DOWN) | ctrl Behavior | What We Are Testing |
|------|----------------|--------|--------------------|---------------|---------------------|
| **0: Reset** | 0 – 4 | 2 | rst = 1 | `ctrl → 32` (init) | **Initialization**: starts at correct value, counters cleared |
| **1: Sustained UP** | 4 – 44 | 20 | UP=1, DOWN=0 | `32 → 37` (+5) | **Filter acceptance**: updates every 4 cycles |
| **2: Idle** | 44 – 64 | 10 | UP=0, DOWN=0 | Hold at 37 | **Idle reset**: no accumulation when inactive |
| **3: Sustained DOWN** | 64 – 104 | 20 | UP=0, DOWN=1 | `37 → 32` (−5) | **Symmetry**: same filtering in reverse |
| **4: Idle** | 104 – 114 | 5 | UP=0, DOWN=0 | Hold at 32 | **Stability**: no change during idle |
| **5: Short UP Burst** | 114 – 120 | 3 | UP=1 | No change (32) | **Glitch rejection**: short bursts ignored |
| **6: Idle Reset** | 120 – 122 | 1 | UP=0, DOWN=0 | Hold at 32 | **Filter clear**: partial counts reset |
| **7: Short UP Burst Again** | 122 – 128 | 3 | UP=1 | No change (32) | **No carry-over**: requires consecutive cycles |
| **8: Sustained UP** | 128 – 168 | 20 | UP=1 | `32 → 37` (+5) | **Recovery**: resumes correct filtering |
| **9: Final Idle** | 168 – 180 | 6 | UP=0, DOWN=0 | Hold at 37 | **Final stability**: holds steady |

**Advantages**
- Reduces jitter near lock  
- Filters out single-cycle noise and glitches  
- Improves stability of the loop  

**Limitations**
- Slower response to real phase error  
- Requires tuning of `FILTER_LEN`  
- Reduced effective loop bandwidth  

Improves steady-state smoothness by filtering noise, at the cost of slower responsiveness.


<br>


#### **Implementation 3: Variable-Step Controller**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/controller/variablestep.png?raw=true" width="500">
</p>

<br>

The **variable-step controller** adapts its step size based on recent behavior.

- Repeated updates in the same direction increase a counter  
- Larger step sizes are applied once thresholds are exceeded  
- Direction changes reset the counter and return to small steps  



##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the Variable-Step Controller (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py controller_variable_step

In [ ]:
# @markdown Click the ▷ button to run your own post-synthesis simulations for this course/fine controller (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py controller_variable_step --process spice

file_name = f"controller_variable_step.csv"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Variable Step Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "controller_variable_step.csv",
    ["CLK", "RST", "UP", "DOWN", "CTRL0", "CTRL1", "CTRL2", "CTRL3", "CTRL4", "CTRL5"]
)

istack(
    layers=[
        {"CLK": traces["CLK"]},
        {"RST": traces["RST"]},
        {"UP": traces["UP"], "DOWN": traces["DOWN"]},
        {"ctrl": decode_ctrl(traces)},
    ],
    title="Varibale Step Controller",
    xlabel="Time (ns)",
    ylabels=["CLK", "RST", "UP / DOWN", "ctrl [0-63]"],
)

**Graph Analysis**
|Phase|Time Range (ns)|Cycles|Inputs (UP / DOWN)|ctrl Behavior|What We Are Testing|
|---|---|---|---|---|---|
|**0: Reset**|0 – 4|2|rst = 1|`ctrl → 32`|**Initialization**: starts at correct value, counters cleared|
|**1: Sustained UP (Acceleration)**|4 – 44|20|UP=1|`32 → 63 (sat)`|**Acceleration**: step ramps 1 → 2 → 4 and saturates at max|
|**2: Idle**|44 – 64|10|UP=0, DOWN=0|Hold at 63|**Idle reset**: holds value and clears direction count|
|**3: Sustained DOWN (Acceleration)**|64 – 104|20|DOWN=1|`63 → lower`|**Down acceleration**: same step ramp in reverse|
|**4: Idle**|104 – 114|5|UP=0, DOWN=0|Hold value|**State reset**: no carry-over of acceleration|
|**5: Alternating Directions**|114 – 134|10|Toggle every 3 cycles|Small ±1 only|**Direction reset**: no acceleration under switching|
|**6: Sustained UP (Restart Acceleration)**|134 – 174|20|UP=1|Accelerates from step=1|**Recovery**: acceleration restarts cleanly|
|**7: Final Idle**|174 – 180|3|UP=0, DOWN=0|Hold value|**Final stability**: no drift at end|

**Advantages**
- Fast convergence when far from lock  
- Automatically adapts without explicit modes  
- Can respond again if the system is disturbed  

**Limitations**
- More parameters to tune (thresholds and step sizes)  
- Risk of overshoot with aggressive steps  
- More complex behavior to analyze and verify  

Provides adaptive behavior that accelerates convergence while maintaining fine resolution near lock, at the cost of increased complexity.

#### **Controller Comparison**

| Controller | Lock Speed | Jitter | Adaptability | Complexity | Key Limitation |
|-----------|-----------|--------|-------------|-----------|----------------|
| Saturating | Slow | High | None | Very Low | Fixed step size |
| Filtered | Medium | Low | Low | Low | Slow response |
| Variable-Step | Fast | Medium-Low | High | Medium | Tuning complexity |



<br>


### **Section 1c: DCDL (Digitally Controlled Delay Line)**

<br>

The DCDL is the actuator of the DLL. It receives the digital control word from the controller and converts it into a physical propagation delay applied to the clock signal. The clock enters a chain of delay elements, and the control word determines how many stages the signal passes through before reaching the output. A larger control value means more active delay stages and a longer delay.

<br>

The DCDL must provide monotonic, glitch-free delay adjustment across its full control range. What varies between the implementations below is the delay primitive (inverters vs. NAND gates), the tap selection strategy (mux trees vs. bypass logic), and how each deals with signal polarity. They span from simple inverter chains to dual-chain vernier interpolation.


#### **Implementation 1: Inverter-Based DCDL**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/4stageinverter.png?raw=true" width="500">
</p>

<br>


The inverter-based DCDL uses cascaded inverters with multiple tap points to generate programmable delay values at each stage. Each tap corresponds to a different propagation delay due to the signal passing though a different number of delay cells at each tap, allowing the output to be selected based on the required timing shift. As shown in the characterization plot above, the delay generally increases in a near-linear stepwise manner as later taps are selected.

<br>

However, because this architecture uses multiplexers to choose between taps, transient glitches may occur during tap switching. This happens because different signal paths experience different propagation delays, so the tap signals may not reach the multiplexer at exactly the same time. In addition, changes in the select lines can briefly cause the wrong tap to be chosen. These effects can generate short incorrect pulses, as seen in the SPICE waveform below.

<br>

These glitches effects have been corrected with a different inverter design called  glitch free inverter dcdl, which we will explain later.

##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the Inverter-Based DCDL (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py inv_dcdl

In [ ]:
# @markdown Run your own post-synthesis simulations for this DCDL (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py inv_dcdl --process spice

file_name = f"inv_dcdl.sp"

labels = ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node", "Q4_node", "Q5_node"]

full_traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    labels,
)

traces = {
    name: (t[::100], v[::100])
    for name, (t, v) in full_traces.items()
}

istack(
    layers=[
        {
            "Control": decode_q(traces)
        },
        {"Input A": traces["A"]},
        {"Output Y": traces["Y"]},
    ],
    title="Inverter DCDL Characterization (64-Stage)",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "Input (V)", "Output (V)"],
    width=1000,
    layer_height=200
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:
labels = ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node", "Q4_node", "Q5_node"]

full_traces = load_wrdata(
    SPICE_DIR / "results" / "inv_dcdl_results.csv",
    labels,
)

traces = {
    name: (t[::100], v[::100])
    for name, (t, v) in full_traces.items()
}


# Glitch from time 500 to 530
istack(
    layers=[
        {
            "Control Value": get_sample(decode_q(traces),500,530)
        },
        {"Input A": get_sample(traces["A"],500,530)},
        {"Output Y": get_sample(traces["Y"],500,530)},
    ],
    title="Inverter DCDL Characterization (64-Stage)",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "Input (V)", "Output (V)"],
    width=1000,
    layer_height=200
)

**Graph Analysis**

In terms of delay performance, the inverter-based DCDL is affected not only by the inverter chain but also by the multiplexer network used for tap selection. Although the taps are generated by inverter stages, the output must propagate through multiplexers, whose delay can be a major contributor to the total path delay. For this architecture structure, the number of mux levels increases approximately as $log_{2}(n)$, where n is the number of stages. As a result, the overall delay may deviate from perfect linearity across the control range.

In [ ]:
# @markdown Example of DCDL working
istack(
    layers=[
        {
            "Control Value": get_sample(decode_q(traces),400,480)
        },
        {"Input A": get_sample(traces["A"],400,480)},
        {"Output Y": get_sample(traces["Y"],400,480)},
    ],
    title="Inverter DCDL Characterization (64-Stage)",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "Input (V)", "Output (V)"],
    width=1000,
    layer_height=200
)


<br>


#### **Implementation 2: Glitch-Free Inverter DCDL**

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/glitchfree.png?raw=true" width="500">
</p>

<br>


This DCDL architecture is an improved version of the inverter-based DCDL designed to eliminate glitches caused by multiplexers. Instead of using a conventional multiplexer tree for tap selection, the selection network is implemented using NAND-gate-based logic. In addition, one-hot encoding is used so that only a single delay path is enabled at any time.

<br>

In a binary-select architecture, multiple control bits must change when switching from one tap to another. Because each control wire has its own propagation delay, these bits do not always transition at exactly the same time. As a result, the selection logic may briefly see an incorrect intermediate code and momentarily enable the wrong path, producing transient pulses or glitches at the output.

<br>

With one-hot encoding, only one control line is asserted for each delay setting, and only one path is allowed to drive the output. During an update, the decoder disables the previous path and enables the new path in a controlled manner, avoiding ambiguous intermediate states caused by skew between multiple control wires. This significantly reduces the switching hazards observed in the original multiplexer-based inverter DCDL.


##### **Run the flow**

In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the Glitchless Inverter-Based DCDL (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py inv_dcdl_glitchless

In [ ]:
# @markdown Run your own post-synthesis simulations for this DCDL (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py inv_dcdl_glitchless --process spice TODO

file_name = f"inv_dcdl_glitchless.sp"

labels = ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node", "Q4_node", "Q5_node"]

full_traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    labels,
)

traces = {
    name: (t[::100], v[::100])
    for name, (t, v) in full_traces.items()
}

istack(
    layers=[
        {
            "Control Value": decode_q(traces)
        },
        {"Input A": traces["A"]},
        {"Output Y": traces["Y"]},
    ],
    title="Glitch-Free Inverter DCDL Characterization (64-Stage)",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "Input (V)", "Output (V)"],
    width=1000,
    layer_height=200
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

labels = [
    "A_in", "Y",
    "Q0_node", "Q1_node", "Q2_node", "Q3_node", "Q4_node", "Q5_node",
    "clk_node", "rst_node"
]

# 2. Load the data (load_wrdata returns a dict of (time, voltage) tuples)
full_traces = load_wrdata(
    SPICE_DIR / "results" / "inv_dcdl_glitchless_results.csv",
    labels
)

# 3. Downsample for faster plotting (1 out of every 100 points)
traces = {
    name: (t[::100], v[::100])
    for name, (t, v) in full_traces.items()
}

# 4. Generate the interactive stacked plot
istack(
    layers=[
        {
            "Control Value": get_sample(decode_q(traces),300,350)
        },
        {"Input A": get_sample(traces["A_in"],300,350)},
        {"Output Y": get_sample(traces["Y"],300,350)},
    ],
    title="Glitchless Inverter DCDL Characterization (64-Stage)",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "Clk / Rst (V)", "Input (V)", "Output (V)"],
    width=1000,
    layer_height=200
)

**Graph Analysis**

As shown in the SPICE simulation waveforms, the output delay remains stable while maintaining glitch-free transitions across the control range.

Another important observation is that the NAND selection network also scales logarithmically with respect to the number of delay cells when implemented as a tree structure. As the number of taps increases, the selected signal must propagate through additional NAND logic stages. Therefore, this delay line may also exhibit some nonlinearity, since the total delay includes not only the inverter-chain delay but also the propagation delay of the NAND selection network.



<br>


#### **Implementation 3: NAND-Based DCDL**

<br>

<p align="center">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/dcdl/nand.png?raw=true" width="500">
</p>

<br>

This is another DCDL architecture that uses the propagation delay of a NAND-gate network to generate the delay of each stage. Because the delay cells are identical across all stages, this delay line exhibits the most linear behavior among the evaluated designs. Each additional stage contributes approximately the same incremental delay, resulting in a more uniform delay response.

This architecture also has the highest delay per stage because NAND gates typically have larger propagation delays than simple inverters due to their increased number of transistors. The consistent square-wave outputs observed in the SPICE simulation are illustrated in the graph below.

Because this architecture uses a larger number of logic gates, it can still introduce small timing mismatches or occasional transient effects. However, the glitch behavior is significantly reduced compared with the multiplexer based inverter DCDL architecture.

##### **Run the flow**


In [ ]:
# @markdown
# @markdown Click the ▷ button to run the flow for the NAND-Based DCDL (neccesary if you want to run custom spice simulations)
# @markdown

!python3 CAC_2026/scripts/flow_cli.py nand_dcdl

In [ ]:
# @markdown Run your own post-synthesis simulations for this DCDL (NOTE: Each spice simulation will take some time to run)

!python CAC_2026/scripts/flow_cli.py nand_dcdl --process spice TODO

file_name = f"nand_dcdl.sp"

traces = load_wrdata(
    SPICE_DIR / "tmp_results" / file_name,
    ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node"],
)

istack(
    layers=[
        {"Control Value": decode_q(traces)},
        {"A": traces["A"]},
        {"Y": traces["Y"]},
    ],
    title="NAND DCDL",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "A (V)", "Y (V)"],
)

In [ ]:
# @markdown If you don't want to wait for spice simulations, view our previously ran results here:

traces = load_wrdata(
    SPICE_DIR / "results" / "nand_dcdl_results.csv",
    ["A", "Y", "Q0_node", "Q1_node", "Q2_node", "Q3_node"],
)

istack(
    layers=[
        {"Control Value": get_sample(decode_q(traces, bits = 4),0,40)},
        {"A": get_sample(traces["A"],0,40)},
        {"Y": get_sample(traces["Y"],0,40)},
    ],
    title="NAND DCDL",
    xlabel="Time (ns)",
    ylabels=["Control Value (0-63)", "A (V)", "Y (V)"],
)

---
## **Section 2: Top Level Zero Delay Buffer Integration**
---

<br>

The Zero-Delay Buffer (ZDB) is a DLL architecture that aligns the output clock with the input clock by compensating for delays introduced by the clock distribution network. Instead of physically removing delay, it adjusts an internal delay line so that the clock observed at the endpoint is phase-aligned with the reference, effectively canceling clock tree insertion delay.

<br>

We selected the ZDB as our primary DLL implementation because it directly addresses one of the most critical challenges in VLSI systems of clock skew and timing alignment. It is widely used in industry, robust to real-world non-idealities, and provides a practical balance between performance, stability, and implementation complexity. This makes it especially well-suited for architectural exploration and realistic system-level modeling.

<br>

Additionally, it is the most direct implementation of the DLL, allowing users to easily view the impact that changes in part architecture and key parameters can have on overall convergence patterns.


<br>


### **Section 2a: Interchangeable Python-Based Components for Architectural Exploration**

<br>

This Python based simulator models the behavior of the different architectural implementations of the DLL parts (Section 1). This includes the phase detectors, controllers, and delay lines. Each implementation uses propagation delay values extracted from our SPICE simulations.

<br>

The simulator allows the user to select different architectures for each block of the DLL. The clock period can be adjusted, and the initial phase relationship between `clk_in` and `clk_out` can be set.

<br>

For manual values, it is important to note that for proper lock acquisition, `clk_out` input value should initially lead or lag `clk_in` by greater than half of the `clk_in` period.

<br>

The phase error represents the time difference between `clk_in` and `clk_out`.
The simulation results can be viewed in the chart. Since all of our DCDL implementations are coarse-grained, the phase error is expected to converge to the nearest achievable delay value and then oscillate around the lock point rather than always reaching exactly zero. This occurs because the desired phase shift may not be an exact multiple of the delay-cell step size. As a result, the phase error may settle at a small offset or periodically cross zero. This behavior can be observed in the phase-error-versus-time graph.

The clock edges versus time graph shows the clock-edge skew lines converging as the DLL approaches lock.


<br>


##### **Python Simulator**

In [ ]:
# @markdown,
# @markdown Click the ▷ button to interact with GUI (configure and run DLL simulations based on characterized data),
# @markdown,
import sys
import importlib
from IPython.display import display

repo_path = "/content/CAC_2026"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

for name in list(sys.modules):
    if name == "simulator" or name.startswith("simulator."):
        del sys.modules[name]

import simulator
import simulator.gui_common
import simulator.notebook_widgets

importlib.reload(simulator.gui_common)
importlib.reload(simulator.notebook_widgets)
importlib.reload(simulator)

from simulator import display_dll_simulator
display(display_dll_simulator())


<br>


### **Section 2b: RTL SimulationDelay Locked Loop (DLL) Parameter Exploration Framework**

<br>

This section introduces an interactive framework for exploring Delay-Locked Loop (DLL) behavior by providing a parameterizable **SystemVerilog behavioral model of a Zero Delay Buffer (ZDB), calibrated using internal delay characteristics derived from our SPICE simulations.

<br>

To complement the hardware model, a Python-based analysis and visualization pipeline is integrated, enabling users to efficiently evaluate system behavior and design trade-offs.

<br>

The goal of this environment is to allow users to:
- Modify key RTL parameters within the DLL submodules
- Simulate and analyze the resulting system behavior
- Observe the impact of design decisions on locking performance and stability
- Receive structured feedback on configuration validity and design quality

<br>

Overall, this platform is designed to facilitate design-space exploration, intuition development, and validation of architectural trade-offs, providing a practical and insightful learning experience for DLL-based VLSI systems.

<br>

---

<br>

**Key Parameters**

<br>

The behavior of the DLL is governed by the following parameters:

<br>

| Parameter         | Description                                                               |
| ----------------- | ------------------------------------------------------------------------- |
| `CLK_PERIOD`      | Reference clock period (ns)                                               |
| `CTRL_BITS`       | Number of control bits; determines number of delay stages (`2^CTRL_BITS`) |
| `INIT_CTRL`       | Initial value of the control variable                                     |
| `DELAY_PS`        | Delay per stage (technology-dependent; e.g., 600 ps or 700 ps)            |
| `UPDATE_DIV_BITS` | Frequency of controller updates: (MIN 2)                                  |

<br>

---

<br>

**Fundamental Constraints**

<br>

1. **Quantization Limit:** The delay line operates in discrete increments. As a result, the phase error cannot be reduced below:

```id="v2r9c7"
≈ ± (DELAY / 2)
```


2. **Delay Range Requirement**

For the DLL to lock, the total available delay must cover the clock period (accounting for tolerance):

```id="m9b2hz"
max_delay = (2^CTRL_BITS) × DELAY ≥ CLK_PERIOD - tolerance
```

If this condition is not satisfied, the system cannot lock under any circumstances.

3. **Lock Tolerance Definition**

Lock is defined as the phase error remaining within a specified tolerance:

```id="g7r1pd"
|phase_error| < tolerance
```

The tolerance is chosen to reflect both physical and system-level constraints:

```id="w8x5qn"
tolerance = min(1.5 × DELAY, 0.25 × CLK_PERIOD)
```

- `1.5 × DELAY` accounts for quantization and controller dynamics
- `0.25 × CLK_PERIOD` prevents overly permissive definitions of lock



##### **Pick inputs here**

<br>


In [ ]:
# @markdown USER INPUTS
CLK_PERIOD = 4 # @param {type:"integer"}
CTRL_BITS = 6 # @param {type:"integer"}
INIT_CTRL = 4 # @param {type:"integer"}
DELAY_PS = 600 # @param {type:"integer"}  # must be 600(inverter cell) or 700(nand cell)
UPDATE_DIV_BITS = 3 # @param {type:"integer"}

In [ ]:
# @markdown Run this to get feedback and generate a testbench with the chosen parameters for our ZDB top.


ZDB_BEHAV_TEST_PATH = PROJECT_ROOT / "design_root/zdb_behav_test"
INPUT_TB = f"{ZDB_BEHAV_TEST_PATH}/tb_zdb_top_behav.sv"
OUTPUT_TB = f"{ZDB_BEHAV_TEST_PATH}/tb_zdb_top_behav_CREATED.sv"


# ===== RUN ANALYZER =====
print("Running config check...\n")

!python3 "{scripts_path}/1_analyze_inputs.py" \
    --clk {CLK_PERIOD} \
    --ctrl {CTRL_BITS} \
    --init {INIT_CTRL} \
    --delay {DELAY_PS} \
    --update {UPDATE_DIV_BITS}


# ===== GENERATE MODIFIED TESTBENCH =====
print("\nGenerating modified testbench...\n")

!python3 "{scripts_path}/2_create_modified_tb.py" {INPUT_TB} \
    --set \
    CLK_PERIOD={CLK_PERIOD} \
    CTRL_BITS={CTRL_BITS} \
    INIT_CTRL={INIT_CTRL} \
    DELAY_PS={DELAY_PS} \
    UPDATE_DIV_BITS={UPDATE_DIV_BITS} \
    -o {OUTPUT_TB}


# ===== VERIFY =====
print("\nGenerated file:")
!ls -lh {OUTPUT_TB}

# ===== Simulate =====
!/usr/bin/iverilog -g2012 -o "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_sim" "{ZDB_BEHAV_TEST_PATH.expanduser()}/tb_zdb_top_behav_CREATED.sv" "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_top_behav.sv" "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_phase_detector_behav.sv" "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_dcdl_behav.sv" "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_controller_behav.sv"
!/usr/bin/vvp "{ZDB_BEHAV_TEST_PATH.expanduser()}/zdb_sim"

In [ ]:
# @markdown Run this to visualize the clock phase error over time
from plot_dll_locking import load_vcd_data, plot_phase_overlay
data = load_vcd_data("zdb.vcd")
plot_phase_overlay(data)

### **RTL to GDS Flow Results**

<br>


<div style="display: flex; justify-content: center; gap: 20px;">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/ZDB_TOP_GDS.png?raw=true" width="400">
  <img src="https://github.com/SiliconJackets/CaC_Spring26/blob/main/images/zdb_top_2.png?raw=true" width="400">
</div>

## **Section 3: Multiphase and TDC-Based DLL Architectures**

Beyond the Zero-Delay Buffer (ZDB), DLLs can also be configured to serve fundamentally different roles in a system such as **phase generation** and **time measurement**. In this section, we introduce two additional architectures: the **Multiphase DLL** and the **Time-to-Digital Converter (TDC)-based DLL**. Users are encouraged to explore their corresponding RTL implementations in the repository to understand how these concepts map to real hardware.

```bash
design_root/src/verilog/tops
```

<br>

---

<br>

### Multiphase DLL: Phase Generation

<br>

A multiphase DLL leverages the delay line not just for alignment, but as a **distributed phase generator**, where intermediate taps provide evenly spaced clock phases. This enables the generation of multiple clock edges within a single period, which is essential for high-speed interfaces and mixed-signal systems.

In practice, this architecture is widely used in applications such as:

- SERDES and high-speed I/O
- ADC/DAC timing  
- Clock interpolation and phase selection

However, the effectiveness of a multiphase DLL is highly sensitive to **delay matching across stages.** Any mismatch in the delay elements directly translates into **non-uniform phase spacing**, which degrades system performance. As highlighted in the textbook, delay lines must maintain consistent resolution and monotonic behavior to ensure predictable phase generation .

<br>

---

<br>

### Time-to-Digital Converter (TDC) DLL: Time Measurement

<br>

In contrast, a TDC-based DLL treats the delay line as a **time quantization structure**, where the phase difference between signals is encoded into a digital value. Rather than simply correcting phase, this architecture enables **direct measurement of timing error**, making it highly useful for calibration and adaptive systems.

Typical use cases include:

- On-chip timing calibration
- Process/voltage/temperature (PVT) tracking
- Fine-grained delay measurement and compensation

This approach is particularly attractive in advanced nodes, where variability dominates and explicit measurement provides better robustness than implicit feedback control. As discussed in the textbook, the resolution and accuracy of such systems depend heavily on delay-cell granularity and the ability to maintain consistent delay steps across operating conditions .

<br>

---

<br>

### Practical Implementation Considerations

While these architectures differ in intent, they share several critical implementation challenges:

- **Delay Line Non-Idealities**  
    Variations in standard cell delays, routing parasitics, and loading can distort both phase generation and time measurement accuracy
    
- **Quantization Effects**  
    Finite delay resolution introduces error floors, leading to phase offset or measurement uncertainty
    
- **Glitch and Hazard Sensitivity**  
    Asynchronous transitions and combinational hazards can corrupt delay selection and phase detection
    
- **PVT Variability**  
    Changes in process, voltage, and temperature directly impact delay characteristics, requiring robust design margins
    

Real DLL behavior is dominated by these physical effects, and architectures that appear equivalent at a high level often diverge after layout due to these constraints .

<br>

---

<br>

### Industry Practices and Standards

In industry, the choice between ZDB, multiphase, and TDC-based DLLs is driven by system requirements rather than theoretical optimality:

- **ZDB DLLs** are the standard for clock deskew in synchronous digital systems
- **Multiphase DLLs** are commonly used in high-speed communication and data conversion systems where multiple precise phases are required
- **TDC-based DLLs** are increasingly used in advanced nodes and calibration-heavy designs, where measurement and adaptability are critical

Modern implementations must also align with standard digital design flows (e.g., synthesizability, timing closure, and clock-tree integration), especially in open-source flows like Sky130. As a result, architectures are often chosen not just for performance, but for **robustness, scalability, and compatibility with physical design constraints**.




---
## **Conclusion**
---

Together, these architectures demonstrate that a DLL is not a single fixed structure, but a flexible timing primitive. By exploring their RTL implementations, users can observe how the same underlying building blocks (phase detection, control logic, and delay lines) can be recomposed to serve very different system-level objectives.
<br>